In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_classic.tools import Tool
from langchain_classic.agents import initialize_agent, AgentType
from langchain_community.tools import DuckDuckGoSearchResults

llm = ChatOllama(
    model="mistral", 
    temperature=0)

# Simple QA tool (LLMChain)
qa_prompt = PromptTemplate.from_template("Answer clearly: {question}")
qa_chain = qa_prompt | llm
qa_tool = Tool(
    name="Simple_QA",
    func=qa_chain.invoke,
    description="Answer the question without hallucination"
)

# Web search tool (DuckDuckGo)
search_tool = Tool(
    name="Web_Search",
    func=DuckDuckGoSearchResults(num_results=3, output_format='list').invoke,
    description="Search the internet for current info"
)

tools = [qa_tool, search_tool]


In [6]:
#1) Zero-Shot ReAct (simple + reliable default)
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    handle_parsing_errors=True, 
    verbose=True
)
result = agent.invoke("Summarize LangChain in 2 lines, then tell me who created it.")



> Entering new AgentExecutor chain...
 I need to find information about LangChain and its creator. I will use Simple_QA to summarize LangChain and then use Web_Search to find out who created it.

Action: Simple_QA
Action Input: 'What is LangChain?'
Observation: content=" LangChain is not a widely known term in the tech industry. However, if we're to infer from the context, it might be a hypothetical or fictional concept that combines language and blockchain technologies. In reality, there isn't an existing service or product named LangChain. If you're referring to a specific LangChain, could you please provide more context or details?" additional_kwargs={} response_metadata={'model': 'mistral', 'created_at': '2026-03-12T10:29:15.9300819Z', 'done': True, 'done_reason': 'stop', 'total_duration': 10735325600, 'load_duration': 32986100, 'prompt_eval_count': 13, 'prompt_eval_duration': 674977300, 'eval_count': 83, 'eval_duration': 9941209700, 'logprobs': None, 'model_name': 'mistral', 'mo

In [11]:
#2) Conversational ReAct (with memory; non-chat model baseline)
from langchain_classic.memory import ConversationBufferMemory

memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

agent = initialize_agent(
    tools=tools,
    llm=llm,
    memory=memory,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    handle_parsing_errors=True,
    verbose=True
)
result = agent.invoke("Summarize LangChain in 2 lines, then tell me who created it.")




> Entering new AgentExecutor chain...
 Thought: Do I need to use a tool? No
AI: LangChain is a platform that allows developers to build and deploy large language models. It was created by Mistral AI.

> Finished chain.


In [12]:
#3) Chat Conversational ReAct (recommended for GPT-3.5/4 chat)
agent = initialize_agent(
    tools=tools,
    llm=llm,
    memory=memory,
    agent=AgentType.CHAT_CONVERSATIONAL_REACT_DESCRIPTION,
    verbose=True
)
# result = agent.invoke("Plan a 3-step study path for LangChain.")
result = agent.invoke("Summarize LangChain in 2 lines, then tell me who created it.")




> Entering new AgentExecutor chain...
 ```json
{
    "action": "Final Answer",
    "action_input": "LangChain is a platform for building and deploying large language models. It was created by Mistral AI."
}
```

> Finished chain.


In [17]:
from langchain_classic.tools import StructuredTool
from langchain_classic.agents import initialize_agent, AgentType
from pydantic import BaseModel

# ✅ Define the input schema using Pydantic
class TitleInput(BaseModel):
    title: str
    tone: str = "concise"

# ✅ Define the tool function
def title_tool_fn(title: str, tone: str = "concise") -> str:
    return f"{tone.title()} Title: {title} in Practice"

# ✅ Create the structured tool
title_tool = StructuredTool.from_function(
    name="TitleMaker",
    func=title_tool_fn,
    description="Generate a title given a topic and an optional tone.",
    args_schema=TitleInput
)

# ✅ Initialize the agent
agent = initialize_agent(
    tools=[title_tool],
    llm=llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

# ✅ Run the agent
result = agent.run("Make a friendly title about LangGraph tutorials.")
print("✅ Result:", result)




> Entering new AgentExecutor chain...
 Action:
```
{
  "action": "TitleMaker",
  "action_input": {
    "title": "Mastering LangGraph: Comprehensive Tutorials",
    "tone": "inviting"
  }
}
```

Observation: Inviting Title: Mastering LangGraph: Comprehensive Tutorials in Practice
Thought: I have generated a friendly title for the LangGraph tutorials. The title is "Mastering LangGraph: Comprehensive Tutorials". If you have any other requirements or need a different tone, feel free to ask!

Action:
```
{
  "action": "Final Answer",
  "action_input": "The title for the LangGraph tutorials is 'Mastering LangGraph: Comprehensive Tutorials'."
}
```



> Finished chain.
✅ Result: The title for the LangGraph tutorials is 'Mastering LangGraph: Comprehensive Tutorials'.


In [21]:
#7) Self-Ask with Search (clarifies, then searches)

# Web search tool (DuckDuckGo)
search_tool = Tool(
    name="Intermediate Answer",  # This name is REQUIRED
    func=DuckDuckGoSearchResults(num_results=3, output_format='list').invoke,
    description="Use this tool to search for missing facts"
)

agent = initialize_agent(
    tools=[search_tool],
    llm=llm,
    agent=AgentType.SELF_ASK_WITH_SEARCH,
    verbose=True,
    handle_parsing_errors=True
)
result = agent.run("When was the first LangChain release and who founded it?")
print(result)



> Entering new AgentExecutor chain...
 Yes, follow-up questions are needed to provide a complete answer.

Follow up: Who founded LangChain?
Intermediate answer: [{'snippet': '3 days ago -In April 2023, LangChain had incorporated and the new startup raised over $20 million in funding at a valuation of at least $200 million from venture firm Sequoia Capital, a week after announcing a $10 million seed investment from Benchmark.', 'title': 'LangChain - Wikipedia', 'link': 'https://en.wikipedia.org/wiki/LangChain'}, {'snippet': '1 day ago -{sports, machine learning, software engineering, statistics} · Experience: LangChain · Education: Harvard University · Location: San Francisco · 500+ connections on LinkedIn. ViewHarrison Chase’s profile on LinkedIn, a professional community ...', 'title': 'Harrison Chase - LangChain | LinkedIn', 'link': 'https://www.linkedin.com/in/harrison-chase-961287118/'}, {'snippet': '5 days ago -LangChain is a series B company based in San Francisco (United State

![Model Diagram](agenttypes1.png)

![Model Diagram](agenttype2.png)

In [ ]:
Deep Dive

![Model Diagram](pro1.png)

![Model Diagram](pro2.png)

![Model Diagram](pro3.png)